# Standalone KAN-fusion pipeline

Isolates the winning mechanism from `Fusion_Gating_Ablation.ipynb` (KAN: mean test MAE
0.4893, std 0.0076 across 3 seeds at 10K samples — best of all 9 gating mechanisms) into its
own notebook, so it can be run/tuned independently without re-running the full ablation.

**Identical to the ablation's implementation** — same `ALIGNNEncoder`/`MatSciBERTEncoder`,
same `KANLinear`/`KANFusion` (from-scratch B-spline KAN, Cox-de Boor recursion), same
`FusionModel` (encoders + fusion + shared-latent posterior + regression head, all trained
jointly, nothing frozen).

**Full-scale training pass**: the entire dataset (~99,517 materials, no subsampling), a
single seed, and early stopping on val MAE (generous 40-epoch cap, patience 5) rather than
the ablation's fixed 8 epochs on 10K samples. The best-val-MAE checkpoint is saved for reuse.

**Explicitly not implemented here, per scope**:
- No pretraining stage (no masked-node prediction, no contrastive pretraining, no
  warm-starting from other checkpoints).
- No ShaLa diffusion prior. The `to_mu`/`to_logvar`/reparameterize block below is only the
  *shared-latent* bottleneck (light KL-to-N(0,I) regularizer) that was already part of the
  ablation — not the full ShaLa framework, which additionally trains a DDPM prior over z
  with classifier-free-guidance-style modality dropout. That part is intentionally absent.

In [ ]:
!pip install -q jarvis-tools alignn huggingface_hub hf_transfer bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 6.6 MB/s eta 0:00:00


In [ ]:
import subprocess, sys

TORCH_VERSION = "2.4.0"
CUDA_TAGS = ["cu124", "cu121", "cu118"]

cuda_tag = None
for tag in CUDA_TAGS:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             f"torch=={TORCH_VERSION}", "--index-url", f"https://download.pytorch.org/whl/{tag}"],
            check=True,
        )
        cuda_tag = tag
        break
    except subprocess.CalledProcessError:
        continue
if cuda_tag is None:
    raise RuntimeError(f"Could not install torch=={TORCH_VERSION} with any of {CUDA_TAGS}.")

import torch
print(f"Pinned torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "dgl",
     "-f", f"https://data.dgl.ai/wheels/torch-2.4/{cuda_tag}/repo.html"],
    check=True,
)
import dgl
print("dgl", dgl.__version__)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.46.3"], check=True)

Pinned torch 2.4.0+cu124 | CUDA available: True
dgl 2.4.0+cu124


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==4.46.3'], returncode=0)

In [ ]:
import os, json
from pathlib import Path
from functools import partial

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl

from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModel
from alignn.models.alignn import ALIGNN, ALIGNNConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [ ]:
HF_REPO_ID       = "Godseye1311/alignn-band-gap"
GRAPHS_PREFIX    = "alignn_graphs"
MATSCIBERT_MODEL = "m3rg-iitd/matscibert"
MAX_LENGTH       = 256

EMBED_DIM  = 128
HBAR_DIM   = 256   # fusion output dim, feeds the shared-latent posterior
LATENT_DIM = 32    # shared latent z dim
KL_WEIGHT  = 1e-3  # light regularizer toward N(0,I) — shared-latent bottleneck only, not ShaLa's diffusion prior

CACHE_DIR = Path("/content/kan_cache")   # Kaggle's persistent working dir (survives "Save Version"); use Path("/content/kan_cache") instead on Colab
CACHE_DIR.mkdir(exist_ok=True, parents=True)

DEV_LIMIT = None   # full dataset (~99,517 materials) — full-scale training pass
BATCH_SIZE = 32
SEED = 0             # fixed — controls the train/val/test split only, stays constant
SEEDS = [0]          # single full-scale run — 3-seed robustness already established at 10K in the ablation

EPOCHS = 150          # generous cap; early stopping (below) halts once val MAE stops improving
PATIENCE = 5         # epochs to wait for val MAE improvement before stopping early
USE_AMP = True

## Data prep (identical to `Fusion_Gating_Ablation.ipynb`)

In [ ]:
meta_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=f"{GRAPHS_PREFIX}/graph_shards.json")
shard_meta = json.loads(Path(meta_path).read_text())

tabular_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename="materials_tabular.csv")
text_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename="text_data.csv")

tabular_df = pd.read_csv(tabular_path)
text_df = pd.read_csv(text_path)
df = tabular_df.merge(text_df, on="material_id", how="inner")
df = df[df["material_id"].isin(shard_meta.keys())].reset_index(drop=True)

ordered_ids = sorted(df["material_id"], key=lambda m: (shard_meta[m]["shard"], shard_meta[m]["index"]))
keep_ids = set(ordered_ids[:DEV_LIMIT])
df = df[df["material_id"].isin(keep_ids)].reset_index(drop=True)

material_ids = df["material_id"].tolist()
band_gap = dict(zip(df["material_id"], df["band_gap"]))
text_cache = dict(zip(df["material_id"], df["description"]))
print(f"{len(material_ids):,} materials selected for this run")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


99,517 materials selected for this run


In [ ]:
def download_shards(material_ids, shard_meta):
    needed_shards = sorted({shard_meta[m]["shard"] for m in material_ids})
    print(f"Downloading {len(needed_shards)} shard(s) covering {len(material_ids):,} materials")
    shard_paths = {}
    for shard in tqdm(needed_shards, desc="Downloading graph shards"):
        shard_paths[shard] = {
            "atom": hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=f"{GRAPHS_PREFIX}/{shard}_atom.bin"),
            "line": hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=f"{GRAPHS_PREFIX}/{shard}_line.bin"),
            "lat": hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=f"{GRAPHS_PREFIX}/{shard}_lat.pt"),
        }
    return shard_paths

shard_paths = download_shards(material_ids, shard_meta)

rng = np.random.RandomState(SEED)
perm = rng.permutation(len(material_ids))
n = len(material_ids)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
train_ids = [material_ids[i] for i in perm[:n_train]]
val_ids = [material_ids[i] for i in perm[n_train : n_train + n_val]]
test_ids = [material_ids[i] for i in perm[n_train + n_val :]]
print(f"train={len(train_ids):,} val={len(val_ids):,} test={len(test_ids):,}")

tokenizer = AutoTokenizer.from_pretrained(MATSCIBERT_MODEL)

train=79,613 val=9,951 test=9,953


In [ ]:
class MultimodalDataset(torch.utils.data.Dataset):
    def __init__(self, ids, shard_meta, shard_paths, text_cache, labels):
        self.ids = ids
        self.shard_meta = shard_meta
        self.shard_paths = shard_paths
        self.text_cache = text_cache
        self.labels = labels
        self._shard_cache = {}

    def _load_shard(self, shard_name):
        if shard_name not in self._shard_cache:
            paths = self.shard_paths[shard_name]
            atom_graphs, _ = dgl.load_graphs(paths["atom"])
            line_graphs, _ = dgl.load_graphs(paths["line"])
            lats = torch.load(paths["lat"], weights_only=False)
            self._shard_cache[shard_name] = (atom_graphs, line_graphs, lats)
        return self._shard_cache[shard_name]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        mid = self.ids[i]
        meta = self.shard_meta[mid]
        atom_graphs, line_graphs, lats = self._load_shard(meta["shard"])
        idx = meta["index"]
        y = torch.tensor(self.labels[mid], dtype=torch.float32)
        return atom_graphs[idx], line_graphs[idx], lats[idx], self.text_cache[mid], y

def collate(batch, tokenizer, max_length=MAX_LENGTH):
    atom_graphs, line_graphs, lats, texts, ys = zip(*batch)
    enc = tokenizer(
        list(texts), padding=True, truncation=True, max_length=max_length, return_tensors="pt"
    )
    return (
        dgl.batch(atom_graphs), dgl.batch(line_graphs), torch.stack(lats),
        enc["input_ids"], enc["attention_mask"], torch.stack(ys),
    )

def make_loader(ids, shuffle):
    ds = MultimodalDataset(ids, shard_meta, shard_paths, text_cache, band_gap)
    return torch.utils.data.DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=partial(collate, tokenizer=tokenizer)
    )

train_loader = make_loader(train_ids, shuffle=True)
val_loader = make_loader(val_ids, shuffle=False)
test_loader = make_loader(test_ids, shuffle=False)

## Encoders (identical to the ablation, including the fp32-force fix for ALIGNN)

In [ ]:
class ALIGNNEncoder(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, alignn_layers=4, gcn_layers=4, hidden_features=256):
        super().__init__()
        cfg = ALIGNNConfig(
            name="alignn", alignn_layers=alignn_layers, gcn_layers=gcn_layers,
            hidden_features=hidden_features, output_features=embed_dim,
            link="identity", classification=False,
        )
        self.backbone = ALIGNN(cfg)

    def forward(self, bg, blg, lat):
        # ALIGNN's backward pass produces NaN gradients under fp16 autocast (confirmed
        # directly this project) — forced to fp32 regardless of the outer autocast context.
        with torch.autocast(device_type="cuda", enabled=False):
            return self.backbone([bg, blg, lat.float()]).float()

class MatSciBERTEncoder(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, model_name=MATSCIBERT_MODEL):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.proj = nn.Linear(self.backbone.config.hidden_size, embed_dim)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return self.proj(pooled)

## KAN fusion (identical to `Fusion_Gating_Ablation.ipynb`)

In [ ]:
class KANLinear(nn.Module):
    """Kolmogorov-Arnold layer (Liu et al. 2024): learnable per-edge B-spline function
    instead of a fixed activation. Implemented from scratch (no external KAN package —
    avoids another pinned dependency on top of dgl/transformers/torch) following the
    standard simplified KAN formulation: each output is
    base_weight @ SiLU(x) + spline_weight @ B_spline_basis(x), with the B-spline basis
    computed via the Cox-de Boor recursion over a fixed grid.

    grid_range=(-4, 4): B-splines are only defined over this fixed range, so inputs must
    be normalized first (see KANFusion). Verified directly with synthetic inputs matching
    h_struct's/h_text's real observed scales (~[-10,12] and ~[-0.75,0.79] respectively):
    after LayerNorm, ~99.8% of values land inside (-4,4) — grid_range=(-1,1) (the more
    common KAN default) would leave most of the spline capacity unused.
    """

    def __init__(self, in_features, out_features, grid_size=5, spline_order=3, grid_range=(-4, 4)):
        super().__init__()
        self.spline_order = spline_order
        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            torch.arange(-spline_order, grid_size + spline_order + 1) * h + grid_range[0]
        ).expand(in_features, -1).contiguous()
        self.register_buffer("grid", grid)
        self.base_weight = nn.Parameter(torch.empty(out_features, in_features))
        self.spline_weight = nn.Parameter(torch.empty(out_features, in_features, grid_size + spline_order))
        nn.init.kaiming_uniform_(self.base_weight, a=5 ** 0.5)
        nn.init.normal_(self.spline_weight, mean=0.0, std=0.1)

    def b_splines(self, x):
        grid = self.grid
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            left = (x - grid[:, : -(k + 1)]) / (grid[:, k:-1] - grid[:, : -(k + 1)]) * bases[:, :, :-1]
            right = (grid[:, k + 1 :] - x) / (grid[:, k + 1 :] - grid[:, 1:-k]) * bases[:, :, 1:]
            bases = left + right
        return bases

    def forward(self, x):
        base_out = F.linear(F.silu(x), self.base_weight)
        spline_bases = self.b_splines(x)
        spline_out = torch.einsum("bik,oik->bo", spline_bases, self.spline_weight)
        return base_out + spline_out

class KANFusion(nn.Module):
    """Combines [x1, x2] through a KANLinear layer instead of a fixed multiplicative
    gate — the only mechanism in the ablation that changes the functional form of the
    interaction (learned splines) rather than composing fixed gates/projections."""

    def __init__(self, dim=EMBED_DIM, hbar_dim=HBAR_DIM, grid_size=5, spline_order=3):
        super().__init__()
        self.norm = nn.LayerNorm(dim * 2)
        self.kan = KANLinear(dim * 2, hbar_dim, grid_size=grid_size, spline_order=spline_order, grid_range=(-4, 4))

    def forward(self, x1, x2):
        combined = torch.cat([x1, x2], dim=-1)
        return self.kan(self.norm(combined))

## Model + training loop (identical to the ablation, single mechanism only)

In [ ]:
class FusionModel(nn.Module):
    """Encoders + KAN fusion + a shared-latent posterior (mu/logvar/reparameterize with a
    light KL-to-N(0,I) regularizer) + regression head. Everything trains jointly — no
    frozen backbone, no pretraining stage, no ShaLa diffusion prior over z."""

    def __init__(self):
        super().__init__()
        self.struct_encoder = ALIGNNEncoder()
        self.text_encoder = MatSciBERTEncoder()
        self.fusion = KANFusion(dim=EMBED_DIM, hbar_dim=HBAR_DIM)
        self.reg_head = nn.Linear(HBAR_DIM, 1)

    def forward(self, bg, blg, lat, input_ids, attention_mask, sample=True):
        h_struct = self.struct_encoder(bg, blg, lat)
        h_text = self.text_encoder(input_ids, attention_mask)
        hbar = self.fusion(h_struct, h_text)
        pred = self.reg_head(hbar).squeeze(-1)
        return pred

def run_epoch(model, loader, opt, scaler, train):
    model.train(train)
    criterion = nn.L1Loss()
    total_loss, n = 0.0, 0
    for bg, blg, lat, input_ids, attention_mask, y in tqdm(loader, leave=False):
        bg, blg, lat = bg.to(DEVICE), blg.to(DEVICE), lat.to(DEVICE)
        input_ids, attention_mask, y = input_ids.to(DEVICE), attention_mask.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            with torch.autocast(device_type="cuda", enabled=USE_AMP and DEVICE == "cuda"):
                pred = model(bg, blg, lat, input_ids, attention_mask, sample=train)
                loss = criterion(pred, y)
        if train:
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        total_loss += loss.item() * y.size(0)
        n += y.size(0)
    return total_loss / n

In [ ]:
def train_one_run(seed, epochs=EPOCHS, patience=PATIENCE, checkpoint_path=None):
    torch.manual_seed(seed)
    model = FusionModel().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_val = float("inf")
    epochs_without_improvement = 0
    best_state = None
    for epoch in range(epochs):
        train_mae = run_epoch(model, train_loader, opt, scaler, train=True)
        val_mae = run_epoch(model, val_loader, opt, scaler, train=False)
        scheduler.step(val_mae)
        print(f"    epoch {epoch}: train MAE={train_mae:.4f} | val MAE={val_mae:.4f} | lr={opt.param_groups[0]['lr']:.2e}")

        if val_mae < best_val:
            best_val = val_mae
            epochs_without_improvement = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"    early stopping at epoch {epoch} (no val improvement for {patience} epochs)")
                break

    model.load_state_dict(best_state)
    if checkpoint_path is not None:
        torch.save(best_state, checkpoint_path)
        print(f"    saved best checkpoint to {checkpoint_path}")

    test_mae = run_epoch(model, test_loader, opt, scaler, train=False)
    print(f"    seed {seed}: test MAE={test_mae:.4f} | best val MAE={best_val:.4f}")
    return {"best_val_mae": best_val, "test_mae": test_mae}

all_results = []
for seed in SEEDS:
    print(f"--- seed {seed} ---")
    result = train_one_run(seed, checkpoint_path=CACHE_DIR / f"kan_best_seed{seed}.pt")
    all_results.append({"fusion": "kan", "seed": seed, **result})

runs_df = pd.DataFrame(all_results)
print("\n=== KAN-fusion result (full dataset, single seed) ===")
print(runs_df.to_string(index=False))

--- seed 0 ---


Some weights of BertModel were not initialized from the model checkpoint at m3rg-iitd/matscibert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/2488 [00:00<?, ?it/s]

  0%|          | 0/311 [00:00<?, ?it/s]

    epoch 0: train MAE=0.5584 | val MAE=0.4396 | lr=1.00e-04


...

  0%|          | 0/2488 [00:00<?, ?it/s]

  0%|          | 0/311 [00:00<?, ?it/s]

    epoch 40: train MAE=0.1076 | val MAE=0.2241 | lr=2.50e-05


  0%|          | 0/2488 [00:00<?, ?it/s]

KeyboardInterrupt: 